# Обнаружение дрифта данных (Data Drift)

**Дрифт** — это изменение статистических свойств данных (целевой переменной или признаков) со временем.

- Ковариатный дрифт (`Covariate Shift`): Изменилось распределение входных признаков P(X).

- Концептуальный дрифт (`Concept Drift`): Изменилось соотношение между признаками и целевой переменной P(Y|X).

Пример: обнаружение с помощью статистических тестов.

Используем библиотеку `scipy.stats` для сравнения распределений тренировочных данных и данных в продакшене.

In [1]:
# test_drift_detection.py
import pandas as pd
from scipy.stats import ks_2samp, chi2_contingency
import numpy as np

def detect_drift_ks(train_column, prod_column, threshold=0.05):
    """
    Обнаруживает дрифт для непрерывного признака с помощью теста Колмогорова-Смирнова.
    H0: Выборки взяты из одного и того же распределения.
    p-value < threshold -> отвергаем H0, есть дрифт.
    """
    statistic, p_value = ks_2samp(train_column, prod_column)
    return p_value < threshold, p_value

def detect_drift_chisquare(train_column, prod_column, threshold=0.05):
    """
    Обнаруживает дрифт для категориального признака с помощью теста хи-квадрат.
    """
    # Создаем таблицы сопряженности
    train_counts = train_column.value_counts().sort_index()
    prod_counts = prod_column.value_counts().sort_index()
    
    # Выравниваем индексы, на случай если в одной выборке нет каких-то категорий
    all_categories = train_counts.index.union(prod_counts.index)
    train_counts = train_counts.reindex(all_categories, fill_value=0)
    prod_counts = prod_counts.reindex(all_categories, fill_value=0)
    
    contingency_table = np.array([train_counts, prod_counts])
    
    statistic, p_value, dof, expected = chi2_contingency(contingency_table)
    return p_value < threshold, p_value

# Пример использования в тесте
def test_for_data_drift():
    # Загружаем эталонные (тренировочные) данные и текущие продакшен-данные
    # На практике train_df будет загружаться из файла, а prod_df из базы данных или стрима
    train_df = pd.read_csv('data/train.csv')
    prod_df = pd.read_csv('data/prod_batch.csv') 
    
    drift_detected = False
    drift_report = []
    
    for column in ['age', 'salary']: # Проверяем только непрерывные признаки
        is_drift, p_val = detect_drift_ks(train_df[column], prod_df[column])
        if is_drift:
            drift_detected = True
            drift_report.append(f"DRIFT in {column}. p-value: {p_val:.5f}")
    
    for column in ['education']: # Проверяем категориальные признаки
        is_drift, p_val = detect_drift_chisquare(train_df[column], prod_df[column])
        if is_drift:
            drift_detected = True
            drift_report.append(f"DRIFT in {column}. p-value: {p_val:.5f}")
            
    # Если обнаружен дрифт, тест "падает", и мы видим отчет
    assert not drift_detected, "Data drift detected:\n" + "\n".join(drift_report)

# Мониторинг в продакшене с Prometheus

**Prometheus** — это система мониторинга и оповещения. Модель в продакшене может экспортировать метрики в Prometheus.

Что мониторить?

1. Прогнозы модели: Распределение предсказанных вероятностей/классов.
2. Входные данные: Статистику по фичам (среднее, медиана) для обнаружения дрифта в реальном времени.
3. Бизнес-метрики: Если есть обратная связь (напр., кликнул ли пользователь на рекомендацию), можно считать Accuracy/Precision онлайн.
4. Технические метрики: Задержка (latency), количество запросов, ошибки.


### Пример исползования:

In [2]:
# pip install prometheus-client

## Код ML-сервиса

Для этого можем создать .py файл и назвать его, например `ml_service.py`. 

Внутри него будет содержаться следующих код:

In [3]:
from prometheus_client import Counter, Histogram, Gauge, start_http_server
import pandas as pd
import numpy as np
import time
import joblib
from datetime import datetime
import logging

# Настройка логирования
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class MLModelWithMonitoring:
    def __init__(self, model_path):
        # Загрузка модели
        self.model = joblib.load(model_path)
        self.model_version = "v1.2.0"
        
        # Инициализация метрик Prometheus
        self._setup_metrics()
        
        # Запуск HTTP-сервера для метрик на порту 8000
        start_http_server(8000)
        logger.info("Prometheus metrics server started on port 8000")
        
        # Референсные данные для обнаружения дрифта
        self.reference_data = pd.read_csv('data/reference_dataset.csv')
        
    def _setup_metrics(self):
        """Инициализация всех метрик Prometheus"""
        
        # Счетчики
        self.predictions_total = Counter(
            'model_predictions_total', 
            'Total number of predictions', 
            ['model_version', 'status', 'feature_group']
        )
        
        self.prediction_errors = Counter(
            'model_prediction_errors_total',
            'Total number of prediction errors',
            ['model_version', 'error_type']
        )
        
        # Гистограммы
        self.prediction_probability = Histogram(
            'model_prediction_probability',
            'Distribution of prediction probabilities',
            ['model_version', 'feature_group'],
            buckets=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
        )
        
        self.prediction_latency = Histogram(
            'model_prediction_latency_seconds',
            'Prediction latency distribution',
            ['model_version'],
            buckets=[0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0]
        )
        
        # Гаужи (метрики с плавающей точкой)
        self.feature_drift_score = Gauge(
            'model_feature_drift_score',
            'Feature drift detection score (p-value)',
            ['model_version', 'feature_name']
        )
        
        self.data_quality_score = Gauge(
            'model_data_quality_score',
            'Data quality score (0-100)',
            ['model_version', 'check_type']
        )
        
        self.model_confidence = Gauge(
            'model_average_confidence',
            'Average prediction confidence',
            ['model_version']
        )
        
    def validate_input_data(self, features):
        """Валидация входных данных"""
        try:
            # Проверка на наличие NaN
            if features.isnull().any().any():
                self.prediction_errors.labels(
                    model_version=self.model_version, 
                    error_type='missing_data'
                ).inc()
                raise ValueError("Input data contains missing values")
                
            # Проверка типов данных
            if not all(features.dtypes == np.float64):
                self.prediction_errors.labels(
                    model_version=self.model_version,
                    error_type='invalid_data_type'
                ).inc()
                raise ValueError("Invalid data types in input")
                
            return True
            
        except Exception as e:
            logger.error(f"Data validation failed: {e}")
            return False
    
    def detect_feature_drift(self, current_features, feature_name):
        """Обнаружение дрифта для одного признака"""
        from scipy.stats import ks_2samp
        
        try:
            reference_feature = self.reference_data[feature_name]
            current_feature = current_features[feature_name]
            
            # Тест Колмогорова-Смирнова
            statistic, p_value = ks_2samp(reference_feature, current_feature)
            
            # Обновляем метрику дрифта
            self.feature_drift_score.labels(
                model_version=self.model_version,
                feature_name=feature_name
            ).set(p_value)
            
            return p_value
            
        except Exception as e:
            logger.error(f"Drift detection failed for {feature_name}: {e}")
            return 1.0  # Если ошибка - считаем дрифта нет
    
    def predict(self, input_data):
        """Основной метод предсказания с мониторингом"""
        start_time = time.time()
        
        try:
            # Валидация входных данных
            if not self.validate_input_data(input_data):
                self.predictions_total.labels(
                    model_version=self.model_version,
                    status='error',
                    feature_group='unknown'
                ).inc()
                return None
            
            # Определяем группу признаков для сегментации метрик
            feature_group = self._determine_feature_group(input_data)
            
            # Обнаружение дрифта для ключевых признаков
            drift_detected = False
            for feature in ['age', 'income', 'credit_score']:
                p_value = self.detect_feature_drift(input_data, feature)
                if p_value < 0.01:  # Порог значимости
                    drift_detected = True
                    logger.warning(f"Drift detected in feature {feature}: p-value = {p_value}")
            
            # Предсказание
            predictions = self.model.predict_proba(input_data)
            probabilities = predictions[:, 1]  # Вероятности положительного класса
            
            # Вычисляем среднюю уверенность модели
            avg_confidence = np.mean(np.max(predictions, axis=1))
            self.model_confidence.labels(model_version=self.model_version).set(avg_confidence)
            
            # Обновляем метрики
            for prob in probabilities:
                self.prediction_probability.labels(
                    model_version=self.model_version,
                    feature_group=feature_group
                ).observe(prob)
            
            latency = time.time() - start_time
            self.prediction_latency.labels(model_version=self.model_version).observe(latency)
            
            self.predictions_total.labels(
                model_version=self.model_version,
                status='success',
                feature_group=feature_group
            ).inc()
            
            logger.info(f"Prediction successful. Avg confidence: {avg_confidence:.3f}, Latency: {latency:.3f}s")
            
            return probabilities
            
        except Exception as e:
            # Логируем ошибку и обновляем метрики
            self.prediction_errors.labels(
                model_version=self.model_version,
                error_type='prediction_failed'
            ).inc()
            
            self.predictions_total.labels(
                model_version=self.model_version,
                status='error',
                feature_group='unknown'
            ).inc()
            
            logger.error(f"Prediction failed: {e}")
            return None
    
    def _determine_feature_group(self, data):
        """Определяет группу признаков для сегментации метрик"""
        avg_income = data['income'].mean()
        if avg_income < 30000:
            return 'low_income'
        elif avg_income < 70000:
            return 'medium_income'
        else:
            return 'high_income'

# Пример использования
if __name__ == "__main__":
    # Инициализация модели с мониторингом
    model = MLModelWithMonitoring('models/random_forest_v1.pkl')
    
    # Эмуляция входящих запросов
    while True:
        # Генерация тестовых данных
        test_data = pd.DataFrame({
            'age': np.random.normal(35, 10, 100),
            'income': np.random.normal(50000, 20000, 100),
            'credit_score': np.random.normal(650, 100, 100)
        })
        
        # Предсказание
        probabilities = model.predict(test_data)
        
        time.sleep(30)  # Ждем 30 секунд между батчами

FileNotFoundError: [Errno 2] No such file or directory: 'models/random_forest_v1.pkl'

*используем любую другую модель*

Конфигурация Prometheus - `prometheus.yml`

```
global:
  scrape_interval: 15s
  evaluation_interval: 15s

rule_files:
  - "alerting_rules.yml"

scrape_configs:
  - job_name: 'ml_model'
    static_configs:
      - targets: ['localhost:8000']
    scrape_interval: 10s
    metrics_path: /metrics

  - job_name: 'prometheus'
    static_configs:
      - targets: ['localhost:9090']

alerting:
  alertmanagers:
    - static_configs:
        - targets:
          - 'localhost:9093'
```

Настройка alert для Prometheus производится в файле с раширением .yaml - `alerting_rules.yml`

Выглядеть может примерно так:

```
groups:
- name: ml_model_alerts
  rules:
  
  - alert: HighFeatureDrift
    expr: model_feature_drift_score < 0.01
    for: 5m
    labels:
      severity: warning
      component: data_quality
    annotations:
      summary: "Data drift detected in feature {{ $labels.feature_name }}"
      description: "Feature {{ $labels.feature_name }} shows significant drift (p-value: {{ $value | humanize }}). Model version: {{ $labels.model_version }}"
  
  - alert: ModelConfidenceDrop
    expr: model_average_confidence < 0.7
    for: 10m
    labels:
      severity: critical
      component: model_performance
    annotations:
      summary: "Model confidence dropped significantly"
      description: "Average model confidence is {{ $value | humanize }} for version {{ $labels.model_version }}"
  
  - alert: HighErrorRate
    expr: rate(model_prediction_errors_total[5m]) / rate(model_predictions_total[5m]) > 0.05
    for: 3m
    labels:
      severity: critical
      component: service_health
    annotations:
      summary: "High prediction error rate detected"
      description: "Error rate is {{ $value | humanizePercentage }} for model version {{ $labels.model_version }}"
  
  - alert: PredictionLatencySpike
    expr: histogram_quantile(0.95, rate(model_prediction_latency_seconds_bucket[5m])) > 1.0
    for: 2m
    labels:
      severity: warning
      component: performance
    annotations:
      summary: "High prediction latency detected"
      description: "95th percentile latency is {{ $value }}s for version {{ $labels.model_version }}"
  
  - alert: NoPredictions
    expr: rate(model_predictions_total[10m]) == 0
    labels:
      severity: critical
      component: service_health
    annotations:
      summary: "No predictions received for 10 minutes"
      description: "Model {{ $labels.model_version }} is not processing any requests"
```

Когда Prometheus обнаруживает, что условие в expr истинно в течение заданного времени (for), он отправляет алерт в системы типа Slack, PagerDuty, Email.

# Мы также можем настроить интеграцию мониторинга с CI/CD

Например, если мы работаем с gitlab, то можем создать файл `.gitlab-ci.yml` с полным циклом тестирования и мониторинга

```
stages:
  - test
  - build
  - deploy
  - monitoring

variables:
  MODEL_VERSION: "v1.2.0-${CI_COMMIT_SHORT_SHA}"

# Стадия тестирования
unit_tests:
  stage: test
  image: python:3.9
  before_script:
    - pip install -r requirements.txt
  script:
    - python -m pytest tests/unit_tests/ -v --junitxml=unit_test_report.xml
    - python -m pytest tests/data_quality_tests/ -v --junitxml=data_quality_report.xml
  artifacts:
    when: always
    reports:
      junit:
        - unit_test_report.xml
        - data_quality_report.xml
  only:
    - merge_requests
    - main

# Тесты на дрифт
drift_tests:
  stage: test
  image: python:3.9
  before_script:
    - pip install -r requirements.txt
  script:
    - python -m pytest tests/drift_tests/ -v --junitxml=drift_test_report.xml
  artifacts:
    when: always
    reports:
      junit: drift_test_report.xml
  only:
    - main

# Сборка Docker-образа с моделью
build_model:
  stage: build
  image: docker:latest
  services:
    - docker:dind
  before_script:
    - docker login -u $CI_REGISTRY_USER -p $CI_REGISTRY_PASSWORD $CI_REGISTRY
  script:
    - docker build -t $CI_REGISTRY_IMAGE:${MODEL_VERSION} .
    - docker push $CI_REGISTRY_IMAGE:${MODEL_VERSION}
  only:
    - main

# Деплой с проверкой метрик
deploy_with_monitoring:
  stage: deploy
  image: alpine:latest
  before_script:
    - apk add --no-cache curl
  script:
    # Деплой модели (пример для Kubernetes)
    - kubectl set image deployment/ml-model ml-model=$CI_REGISTRY_IMAGE:${MODEL_VERSION}
    
    # Ждем запуска пода и проверяем метрики
    - |
      echo "Waiting for model to start..."
      sleep 30
      
      # Проверяем, что метрики доступны
      MAX_RETRIES=10
      retry_count=0
      
      while [ $retry_count -lt $MAX_RETRIES ]; do
        if curl -s http://ml-model-service:8000/metrics | grep -q "model_predictions_total"; then
          echo "Metrics endpoint is working"
          break
        else
          echo "Waiting for metrics endpoint... ($((retry_count + 1))/$MAX_RETRIES)"
          sleep 10
          retry_count=$((retry_count + 1))
        fi
      done
      
      if [ $retry_count -eq $MAX_RETRIES ]; then
        echo "Metrics endpoint not available after $MAX_RETRIES retries"
        exit 1
      fi
  environment:
    name: production
    url: http://ml-model-service
  only:
    - main

# Пост-деплойная проверка мониторинга
monitoring_validation:
  stage: monitoring
  image: alpine:latest
  before_script:
    - apk add --no-cache curl jq
  script:
    - |
      # Проверяем базовые метрики в Prometheus после деплоя
      echo "Validating monitoring setup..."
      
      # Ждем пока Prometheus соберет первые метрики
      sleep 60
      
      # Проверяем, что метрики поступают в Prometheus
      PROMETHEUS_URL="http://prometheus:9090/api/v1/query"
      
      check_metric() {
        local metric_name=$1
        local query_result=$(curl -s "$PROMETHEUS_URL" --data-urlencode "query=$metric_name" | jq '.data.result | length')
        
        if [ "$query_result" -gt 0 ]; then
          echo "Metric $metric_name is available in Prometheus"
          return 0
        else
          echo "Metric $metric_name not found in Prometheus"
          return 1
        fi
      }
      
      # Проверяем ключевые метрики
      check_metric "model_predictions_total"
      check_metric "model_prediction_probability_bucket"
      check_metric "model_feature_drift_score"
      
      # Проверяем, что нет критических алертов
      ALERTS_JSON=$(curl -s "$PROMETHEUS_URL" --data-urlencode 'query=ALERTS{alertstate="firing", severity="critical"}')
      CRITICAL_ALERTS=$(echo "$ALERTS_JSON" | jq '.data.result | length')
      
      if [ "$CRITICAL_ALERTS" -eq 0 ]; then
        echo "No critical alerts firing"
      else
        echo "Critical alerts detected:"
        echo "$ALERTS_JSON" | jq '.data.result[] | .metric.alertname'
        exit 1
      fi
  dependencies:
    - deploy_with_monitoring
  only:
    - main
```

# Также мы можем настроить дашборд Grafana для визуализации

Для этого создадим Json файл `grafana.json`

```
{
  "dashboard": {
    "title": "ML Model Monitoring",
    "panels": [
      {
        "title": "Prediction Rate",
        "type": "stat",
        "targets": [
          {
            "expr": "rate(model_predictions_total[5m])",
            "legendFormat": "{{status}}"
          }
        ]
      },
      {
        "title": "Feature Drift Detection",
        "type": "heatmap",
        "targets": [
          {
            "expr": "model_feature_drift_score",
            "legendFormat": "{{feature_name}}"
          }
        ]
      },
      {
        "title": "Prediction Probability Distribution",
        "type": "histogram",
        "targets": [
          {
            "expr": "histogram_quantile(0.95, rate(model_prediction_probability_bucket[5m]))"
          }
        ]
      }
    ]
  }
}
```

# Итог:

Prometheus это по сути наша основа мониторинга, в рамках данной демонстрации. 

На нашем примере есть четыре типа метрик:

- Counter - для подсчета событий (количество предсказаний, ошибок)
- Gauge - для значений, которые могут увеличиваться и уменьшаться (уверенность модели, p-value дрифта)
- Histogram - для распределений (вероятности, задержки)
- Summary - аналогично гистограмме, но на стороне клиента

После деплоя автоматически проверяем, что метрики доступны и нет критических алертов. Это доостигается при помощи CI/CD с валидацией мониторинга

Помимо прочего реализуем многоуровневые алерты: от предупреждений о дрифте до критических алертов об остановке сервиса.

Такой подход обеспечивает полную наблюдаемость за ML-моделью на всех этапах её жизненного цикла.